# 通用多因子相关性与固定权重组合检验

本 Notebook 严格隔离训练集与验证集，并支持任意数量因子。以后新增因子时，只需在下一节的 `FACTOR_CONFIGS` 中复制一个配置块，填写因子名称、列名、方向、固定权重以及训练/验证 Parquet 路径；其余单元无需修改。

分析内容会自动扩展为：

- 所有因子的每日横截面 Pearson/Spearman 两两相关性；
- 多因子相关矩阵、年度稳定性、头尾组合重合率；
- 每个单因子、固定权重全组合、逐因子剔除组合的共同口径严格回测；
- IC/策略收益相关矩阵，以及每个因子的边际夏普贡献；
- 训练集与验证集结果自动分目录保存和对比。

验证阶段只能检验训练阶段已经冻结的因子方向、权重、持有期与门槛，不能查看验证结果后再调参。

## 1. 用户唯一修改区、路径与冻结参数

正常切换训练集/验证集只改 `SAMPLE_STAGE`。新增因子只在 `FACTOR_CONFIGS` 中增加一个配置块。

In [ ]:
from pathlib import Path
from IPython.display import display
from itertools import combinations
import gc
import json
import re
import sys

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:.6f}')

LEARNING_DIR = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'Back_test').exists() and (p / 'Factor').exists()), Path.cwd())
WORK_DIR = LEARNING_DIR / 'Correlation_test'

# ============================================================================
# ============================ 用户唯一修改区 ================================
# ============================================================================

# 通常只改这一行：先运行 train，冻结结论后再切换 validation。
SAMPLE_STAGE = 'validation'  # 'train' 或 'validation'

# 样本边界。训练、验证和下一阶段必须严格隔离。
SAMPLE_PROFILES = {
    'train': {
        'label': '训练集',
        'start_date': '2006-01-04',
        'end_date': '2017-12-29',
        'next_start_locked': '2018-01-02',
    },
    'validation': {
        'label': '验证集',
        'start_date': '2018-01-02',
        'end_date': '2021-12-31',
        'next_start_locked': '2022-01-04',
    },
}

# 因子注册表：以后新增因子，只复制一个配置块并填写下面6项。
# direction 必须是 positive/negative；weight 是训练前冻结的非负权重，
# 程序会自动归一化。所有因子会先统一成“越大越看多”再计算相关性和组合。
FACTOR_CONFIGS = {
    'turnover_stability': {
        'column': 'turnover_stability_factor',
        'direction': 'negative',
        'weight': 1.0,
        'train_path': (
            LEARNING_DIR / 'Factor' / 'turnover_stability_output_20060104_20171229'
            / 'turnover_stability_factor_20060104_20171229.parquet'
        ),
        'validation_path': (
            LEARNING_DIR / 'Validation_test' / 'data'
            / 'turnover_stability_output_20180102_20211231'
            / 'turnover_stability_factor_20180102_20211231.parquet'
        ),
    },
    'amihud_20': {
        'column': 'amihud_20_factor',
        'direction': 'positive',
        'weight': 1.0,
        'train_path': (
            LEARNING_DIR / 'Factor' / 'amihud_20_output_20060104_20171229'
            / 'amihud_20_factor_20060104_20171229.parquet'
        ),
        'validation_path': (
            LEARNING_DIR / 'Validation_test' / 'data'
            / 'amihud_20_output_20180102_20211231'
            / 'amihud_20_factor_20180102_20211231.parquet'
        ),
    },
    'volume_price_divergence': {
        'column': 'volume_price_divergence_factor',
        'direction': 'positive',
        'weight': 1.0,
        'train_path': (
            LEARNING_DIR / 'Factor' / 'volume_price_divergence_output_20060104_20171229'
            / 'volume_price_divergence_factor_20060104_20171229.parquet'
        ),
        'validation_path': (
            LEARNING_DIR / 'Validation_test' / 'data'
            / 'volume_price_divergence_output_20180102_20211231'
            / 'volume_price_divergence_factor_20180102_20211231.parquet'
        ),
    },
}

# 所有因子共同采用的冻结回测口径。
HORIZON = 5
EXECUTION_LAG = 1
REBALANCE_STEP = 5
GROUPS = 10
COST_BPS = 10.0
MIN_STOCKS = 100
MAX_MISSING_RETURN_RATIO = 0.40
PRICE_TOLERANCE = 0.005
SAVE_HOLDINGS = False

# 多因子解释门槛与可选计算。
MAX_ABS_MEAN_SPEARMAN = 0.70
MIN_SHARPE_GAIN_FOR_STRONG = 0.05
RUN_LEAVE_ONE_OUT = True  # 三因子以上自动检验“剔除某因子”后的边际贡献

# 首次运行、因子集合变化或行情更新后保持 True；确认缓存有效后可改 False。
BUILD_STRICT_FLAGS = True
RUN_STRICT_BACKTEST = True

# ============================================================================
# ========================== 用户修改区结束 ==================================
# ============================================================================

assert SAMPLE_STAGE in SAMPLE_PROFILES, "SAMPLE_STAGE 只能是 'train' 或 'validation'"
assert len(FACTOR_CONFIGS) >= 2, '相关性分析至少需要两个因子'
assert HORIZON == REBALANCE_STEP, '非重叠回测要求 HORIZON == REBALANCE_STEP'

profile = SAMPLE_PROFILES[SAMPLE_STAGE]
SAMPLE_LABEL = profile['label']
SAMPLE_START_DATE = profile['start_date']
SAMPLE_END_DATE = profile['end_date']
NEXT_START_LOCKED = profile['next_start_locked']
SAMPLE_START_TS = pd.Timestamp(SAMPLE_START_DATE)
SAMPLE_END_TS = pd.Timestamp(SAMPLE_END_DATE)
NEXT_START_LOCKED_TS = pd.Timestamp(NEXT_START_LOCKED)
assert SAMPLE_START_TS <= SAMPLE_END_TS < NEXT_START_LOCKED_TS

FACTOR_KEYS = list(FACTOR_CONFIGS)
assert len(FACTOR_KEYS) == len(set(FACTOR_KEYS)), '因子名称重复'
reserved_names = {'combined_fixed_weight'}
for factor_key, cfg in FACTOR_CONFIGS.items():
    assert re.fullmatch(r'[A-Za-z][A-Za-z0-9_]*', factor_key), (
        f'因子键必须是英文/数字/下划线且以字母开头：{factor_key}'
    )
    assert factor_key not in reserved_names, f'因子名称与保留名冲突：{factor_key}'
    assert cfg['direction'] in {'positive', 'negative'}, f'{factor_key} direction 配置错误'
    assert float(cfg['weight']) >= 0, f'{factor_key} weight 不能为负数'
    for stage in SAMPLE_PROFILES:
        assert f'{stage}_path' in cfg, f'{factor_key} 缺少 {stage}_path'
    cfg['sign'] = 1.0 if cfg['direction'] == 'positive' else -1.0
    cfg['alpha_column'] = f'{factor_key}_alpha'
    cfg['active_path'] = Path(cfg[f'{SAMPLE_STAGE}_path'])

raw_weights = pd.Series({key: float(cfg['weight']) for key, cfg in FACTOR_CONFIGS.items()})
assert raw_weights.sum() > 0, '至少一个因子的 weight 必须大于0'
NORMALIZED_WEIGHTS = raw_weights / raw_weights.sum()

DATE_TAG = f'{SAMPLE_START_TS:%Y%m%d}_{SAMPLE_END_TS:%Y%m%d}'
FACTOR_SET_NAME = '__'.join(FACTOR_KEYS)
COMBINED_COLUMN = 'combined_fixed_weight'
DATA_DIR = WORK_DIR / 'data'
OUTPUT_ROOT = WORK_DIR / 'outputs' / FACTOR_SET_NAME
OUTPUT_DIR = OUTPUT_ROOT / f'{HORIZON}d_{SAMPLE_STAGE}_{DATE_TAG}'
MULTIFACTOR_PATH = DATA_DIR / f'{FACTOR_SET_NAME}_{SAMPLE_STAGE}_{DATE_TAG}.parquet'
FLAGS_DIR = (
    WORK_DIR / '.cache' / 'strict_tradability_flags' / FACTOR_SET_NAME
    / f'{HORIZON}d_{SAMPLE_STAGE}_{DATE_TAG}'
)

BACKTEST_DIR = LEARNING_DIR / 'Back_test'
MARKET_PARQUET = LEARNING_DIR / 'Data' / 'stocks_daily_04-25.parquet'
MARKET_CACHE_DIR = BACKTEST_DIR / '.cache' / 'market_data'

required_paths = [cfg['active_path'] for cfg in FACTOR_CONFIGS.values()]
required_paths += [MARKET_PARQUET, BACKTEST_DIR / 'backtest.py']
for path in required_paths:
    assert path.exists(), f'找不到文件：{path}'
for folder in [DATA_DIR, OUTPUT_DIR, FLAGS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
if str(BACKTEST_DIR) not in sys.path:
    sys.path.insert(0, str(BACKTEST_DIR))

factor_registry = pd.DataFrame([
    {
        'factor': key,
        'column': cfg['column'],
        'direction': cfg['direction'],
        'alignment_sign': cfg['sign'],
        'normalized_weight': NORMALIZED_WEIGHTS[key],
        'active_path': str(cfg['active_path']),
    }
    for key, cfg in FACTOR_CONFIGS.items()
]).set_index('factor')
factor_registry.to_csv(OUTPUT_DIR / 'factor_registry.csv')

frozen_config = pd.Series({
    '当前阶段': f'{SAMPLE_STAGE}（{SAMPLE_LABEL}）',
    '分析区间': f'{SAMPLE_START_DATE} 至 {SAMPLE_END_DATE}',
    '下一阶段锁定起点': NEXT_START_LOCKED,
    '因子数量': len(FACTOR_KEYS),
    '因子集合': ', '.join(FACTOR_KEYS),
    '统一信号/成交/持有': f'T日收盘 / T+{EXECUTION_LAG} VWAP / {HORIZON}交易日',
    '调仓步长': REBALANCE_STEP,
    '交易成本bps': COST_BPS,
    '逐因子剔除检验': RUN_LEAVE_ONE_OUT,
    '输出目录': str(OUTPUT_DIR),
})
display(frozen_config.to_frame('冻结值'))
display(factor_registry)

## 2. 对齐任意数量因子、统一方向并生成固定权重组合

程序按年度读取每个因子，严格使用 `date + code` 内连接。每个因子按配置统一为“越大越看多”，全组合与逐因子剔除组合均在每日横截面重新标准化。

In [ ]:
FACTOR_ALPHA_COLUMNS = {
    key: cfg['alpha_column'] for key, cfg in FACTOR_CONFIGS.items()
}
PAIR_KEYS = list(combinations(FACTOR_KEYS, 2))

# 组合权重完全由修改区决定；逐因子剔除组合只用于边际贡献诊断。
COMBINATION_WEIGHTS = {
    COMBINED_COLUMN: NORMALIZED_WEIGHTS.to_dict(),
}
LEAVE_ONE_OUT_COLUMNS = {}
if RUN_LEAVE_ONE_OUT and len(FACTOR_KEYS) >= 3:
    for omitted in FACTOR_KEYS:
        remaining = raw_weights.drop(index=omitted)
        assert remaining.sum() > 0, f'剔除 {omitted} 后剩余权重为0'
        column = f'combined_without__{omitted}'
        LEAVE_ONE_OUT_COLUMNS[omitted] = column
        COMBINATION_WEIGHTS[column] = (remaining / remaining.sum()).to_dict()

SIGNAL_COLUMNS = list(FACTOR_ALPHA_COLUMNS.values()) + list(COMBINATION_WEIGHTS)
MULTIFACTOR_SCHEMA = pa.schema(
    [('date', pa.timestamp('ns')), ('code', pa.string())]
    + [(column, pa.float64()) for column in SIGNAL_COLUMNS]
    + [('asof_date', pa.timestamp('ns'))]
)

def load_factor_year(factor_key, year):
    cfg = FACTOR_CONFIGS[factor_key]
    factor_column = cfg['column']
    alpha_column = cfg['alpha_column']
    start = max(pd.Timestamp(year=year, month=1, day=1), SAMPLE_START_TS)
    end = min(pd.Timestamp(year=year, month=12, day=31), SAMPLE_END_TS)
    output_columns = ['date', 'code', alpha_column, f'asof__{factor_key}']
    if start > end:
        return pd.DataFrame(columns=output_columns)
    frame = pd.read_parquet(
        cfg['active_path'],
        columns=['date', 'code', factor_column, 'asof_date'],
        filters=[[('date', '>=', start), ('date', '<=', end)]],
    )
    frame['date'] = pd.to_datetime(frame['date'], errors='coerce')
    frame['asof_date'] = pd.to_datetime(frame['asof_date'], errors='coerce')
    frame[factor_column] = pd.to_numeric(frame[factor_column], errors='coerce')
    frame = frame.dropna(subset=['date', 'code', factor_column, 'asof_date'])
    frame = frame[frame['date'].between(SAMPLE_START_TS, SAMPLE_END_TS)].copy()
    assert (frame['asof_date'] <= frame['date']).all(), f'{factor_key} 存在未来数据'
    assert not frame.duplicated(['date', 'code']).any(), f'{factor_key} 存在重复 date-code'
    frame[alpha_column] = cfg['sign'] * frame[factor_column]
    frame = frame.rename(columns={'asof_date': f'asof__{factor_key}'})
    return frame[output_columns]

def daily_pairwise_diagnostics(frame):
    alpha_columns = [FACTOR_ALPHA_COLUMNS[key] for key in FACTOR_KEYS]
    rows = []
    for date, daily in frame.groupby('date', sort=True):
        values = daily[alpha_columns].to_numpy(dtype=float)
        n_common = values.shape[0]
        if n_common < MIN_STOCKS:
            continue
        pearson = np.corrcoef(values, rowvar=False)
        ranks = pd.DataFrame(values).rank(axis=0, pct=True).to_numpy(dtype=float)
        spearman = np.corrcoef(ranks, rowvar=False)
        long_flags = ranks > 0.90
        short_flags = ranks <= 0.10
        for i, j in combinations(range(len(FACTOR_KEYS)), 2):
            long_intersection = int((long_flags[:, i] & long_flags[:, j]).sum())
            long_union = int((long_flags[:, i] | long_flags[:, j]).sum())
            short_intersection = int((short_flags[:, i] & short_flags[:, j]).sum())
            short_union = int((short_flags[:, i] | short_flags[:, j]).sum())
            long_min = min(int(long_flags[:, i].sum()), int(long_flags[:, j].sum()))
            short_min = min(int(short_flags[:, i].sum()), int(short_flags[:, j].sum()))
            factor_a, factor_b = FACTOR_KEYS[i], FACTOR_KEYS[j]
            rows.append({
                'date': date,
                'factor_a': factor_a,
                'factor_b': factor_b,
                'pair': f'{factor_a}__{factor_b}',
                'pearson_corr': pearson[i, j],
                'spearman_corr': spearman[i, j],
                'n_common': n_common,
                'long_overlap_min': long_intersection / long_min if long_min else np.nan,
                'long_jaccard': long_intersection / long_union if long_union else np.nan,
                'short_overlap_min': short_intersection / short_min if short_min else np.nan,
                'short_jaccard': short_intersection / short_union if short_union else np.nan,
            })
    return pd.DataFrame(rows)

temporary_path = MULTIFACTOR_PATH.with_suffix('.tmp.parquet')
if temporary_path.exists():
    temporary_path.unlink()
writer = pq.ParquetWriter(temporary_path, MULTIFACTOR_SCHEMA, compression='zstd')
diagnostic_frames = []
common_factor_dates = []
total_rows = 0
try:
    for year in range(SAMPLE_START_TS.year, SAMPLE_END_TS.year + 1):
        aligned = None
        asof_columns = []
        for factor_key in FACTOR_KEYS:
            current = load_factor_year(factor_key, year)
            asof_column = f'asof__{factor_key}'
            asof_columns.append(asof_column)
            if aligned is None:
                aligned = current
            else:
                aligned = aligned.merge(
                    current, on=['date', 'code'], how='inner', validate='one_to_one'
                )
        assert aligned is not None and len(aligned) > 0, f'{year} 年因子没有共同样本'

        # 相关性和共同回测只保留横截面数量足够的日期。
        daily_counts = aligned.groupby('date', sort=False).size()
        valid_dates = daily_counts[daily_counts >= MIN_STOCKS].index
        aligned = aligned[aligned['date'].isin(valid_dates)].copy()
        assert len(aligned) > 0, f'{year} 年没有满足 MIN_STOCKS 的共同样本'

        for combination_column, weights in COMBINATION_WEIGHTS.items():
            raw_combined = sum(
                weight * aligned[FACTOR_ALPHA_COLUMNS[key]]
                for key, weight in weights.items()
            )
            daily_mean = raw_combined.groupby(aligned['date'], sort=False).transform('mean')
            daily_std = raw_combined.groupby(aligned['date'], sort=False).transform('std')
            aligned[combination_column] = (
                (raw_combined - daily_mean) / daily_std.where(daily_std > 0)
            )

        aligned['asof_date'] = aligned[asof_columns].max(axis=1)
        aligned = aligned.dropna(subset=SIGNAL_COLUMNS + ['asof_date'])
        assert (aligned['asof_date'] <= aligned['date']).all(), '组合因子存在未来数据'
        assert not aligned.duplicated(['date', 'code']).any()

        diagnostic_frames.append(daily_pairwise_diagnostics(aligned))
        common_factor_dates.extend(pd.to_datetime(aligned['date'].unique()).tolist())
        panel = aligned[['date', 'code'] + SIGNAL_COLUMNS + ['asof_date']].copy()
        panel['code'] = panel['code'].astype(str)
        writer.write_table(
            pa.Table.from_pandas(panel, schema=MULTIFACTOR_SCHEMA, preserve_index=False)
        )
        total_rows += len(panel)
        print(f'{year}: 对齐 {len(panel):,} 行，{panel["date"].nunique()} 个交易日')
        del aligned, panel
        gc.collect()
finally:
    writer.close()

temporary_path.replace(MULTIFACTOR_PATH)
pairwise_daily = pd.concat(diagnostic_frames, ignore_index=True).sort_values(
    ['date', 'factor_a', 'factor_b']
)
common_factor_dates = sorted(set(pd.Timestamp(date) for date in common_factor_dates))
assert not pairwise_daily.empty, '没有生成两两相关性结果'
assert min(common_factor_dates) >= SAMPLE_START_TS
assert max(common_factor_dates) <= SAMPLE_END_TS < NEXT_START_LOCKED_TS
assert pq.ParquetFile(MULTIFACTOR_PATH).metadata.num_rows == total_rows

combination_weight_table = pd.DataFrame(COMBINATION_WEIGHTS).fillna(0.0)
combination_weight_table.index.name = 'factor'
combination_weight_table.to_csv(OUTPUT_DIR / 'combination_weights.csv')
print(f'{SAMPLE_LABEL}多因子面板：', MULTIFACTOR_PATH)
print('因子数量：', len(FACTOR_KEYS), '两两组合：', len(PAIR_KEYS))
print('总记录数：', f'{total_rows:,}')
display(combination_weight_table)

## 3. 全部因子的两两横截面相关性与头尾持仓重合

门槛作用于“各因子对每日 Spearman 的时间均值”。只要任意一对超过冻结阈值，整体相关性筛选就不通过，但仍会保留全部明细用于判断冗余来源。

In [ ]:
pairwise_daily.to_csv(OUTPUT_DIR / 'daily_pairwise_diagnostics.csv', index=False)

pairwise_yearly = (
    pairwise_daily.assign(year=pd.to_datetime(pairwise_daily['date']).dt.year)
    .groupby(['year', 'factor_a', 'factor_b', 'pair'], as_index=False)
    .agg(
        mean_spearman=('spearman_corr', 'mean'),
        median_spearman=('spearman_corr', 'median'),
        std_spearman=('spearman_corr', 'std'),
        mean_pearson=('pearson_corr', 'mean'),
        mean_long_overlap=('long_overlap_min', 'mean'),
        mean_short_overlap=('short_overlap_min', 'mean'),
        mean_common_stocks=('n_common', 'mean'),
        observations=('spearman_corr', 'count'),
    )
)
pairwise_yearly.to_csv(OUTPUT_DIR / 'yearly_pairwise_diagnostics.csv', index=False)

pairwise_summary = (
    pairwise_daily.groupby(['factor_a', 'factor_b', 'pair'], as_index=False)
    .agg(
        mean_spearman=('spearman_corr', 'mean'),
        median_spearman=('spearman_corr', 'median'),
        std_spearman=('spearman_corr', 'std'),
        spearman_5pct=('spearman_corr', lambda x: x.quantile(0.05)),
        spearman_95pct=('spearman_corr', lambda x: x.quantile(0.95)),
        mean_pearson=('pearson_corr', 'mean'),
        mean_common_stocks=('n_common', 'mean'),
        mean_long_overlap_min=('long_overlap_min', 'mean'),
        mean_long_jaccard=('long_jaccard', 'mean'),
        mean_short_overlap_min=('short_overlap_min', 'mean'),
        mean_short_jaccard=('short_jaccard', 'mean'),
        observations=('spearman_corr', 'count'),
    )
)
pairwise_summary['abs_mean_spearman'] = pairwise_summary['mean_spearman'].abs()
pairwise_summary['correlation_pass'] = (
    pairwise_summary['abs_mean_spearman'] <= MAX_ABS_MEAN_SPEARMAN
)
pairwise_summary = pairwise_summary.sort_values('abs_mean_spearman', ascending=False)
pairwise_summary.to_csv(OUTPUT_DIR / 'pairwise_correlation_summary.csv', index=False)

def build_symmetric_matrix(summary, value_column, diagonal=1.0):
    matrix = pd.DataFrame(np.nan, index=FACTOR_KEYS, columns=FACTOR_KEYS, dtype=float)
    for factor_key in FACTOR_KEYS:
        matrix.loc[factor_key, factor_key] = diagonal
    for row in summary.itertuples(index=False):
        matrix.loc[row.factor_a, row.factor_b] = getattr(row, value_column)
        matrix.loc[row.factor_b, row.factor_a] = getattr(row, value_column)
    return matrix

mean_spearman_matrix = build_symmetric_matrix(pairwise_summary, 'mean_spearman')
mean_pearson_matrix = build_symmetric_matrix(pairwise_summary, 'mean_pearson')
long_overlap_matrix = build_symmetric_matrix(pairwise_summary, 'mean_long_overlap_min')
short_overlap_matrix = build_symmetric_matrix(pairwise_summary, 'mean_short_overlap_min')
mean_spearman_matrix.to_csv(OUTPUT_DIR / 'mean_daily_spearman_matrix.csv')
mean_pearson_matrix.to_csv(OUTPUT_DIR / 'mean_daily_pearson_matrix.csv')
long_overlap_matrix.to_csv(OUTPUT_DIR / 'mean_long_overlap_matrix.csv')
short_overlap_matrix.to_csv(OUTPUT_DIR / 'mean_short_overlap_matrix.csv')

correlation_pass = bool(pairwise_summary['correlation_pass'].all())
worst_pair_row = pairwise_summary.iloc[0]
factor_corr_summary = pd.Series({
    'sample_stage': SAMPLE_STAGE,
    'sample_start': SAMPLE_START_DATE,
    'sample_end': SAMPLE_END_DATE,
    'factor_count': len(FACTOR_KEYS),
    'pair_count': len(PAIR_KEYS),
    'worst_pair': worst_pair_row['pair'],
    'max_abs_mean_spearman': worst_pair_row['abs_mean_spearman'],
    'mean_abs_pairwise_spearman': pairwise_summary['abs_mean_spearman'].mean(),
    'minimum_common_stocks': pairwise_daily['n_common'].min(),
    'correlation_threshold': MAX_ABS_MEAN_SPEARMAN,
    'correlation_pass': correlation_pass,
})
factor_corr_summary.to_csv(OUTPUT_DIR / 'factor_correlation_summary.csv', header=['value'])
display(pairwise_summary)
display(mean_spearman_matrix)
display(factor_corr_summary.to_frame('value'))
print('多因子相关性筛选：', '通过' if correlation_pass else '未通过')

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager

CJK_FONT_CANDIDATES = [
    'Hiragino Sans GB', 'Arial Unicode MS', 'Heiti SC',
    'Songti SC', 'PingFang SC', 'Noto Sans CJK SC', 'SimHei',
]
available_font_names = {font.name for font in font_manager.fontManager.ttflist}
CJK_FONT = next((name for name in CJK_FONT_CANDIDATES if name in available_font_names), None)
if CJK_FONT is not None:
    plt.rcParams['font.sans-serif'] = [CJK_FONT, 'DejaVu Sans']
    print('图表中文字体：', CJK_FONT)
else:
    print('未检测到中文字体；计算不受影响，但中文图题可能显示为方框。')
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
heat = axes[0, 0].imshow(mean_spearman_matrix.values, cmap='coolwarm', vmin=-1, vmax=1)
axes[0, 0].set_xticks(range(len(FACTOR_KEYS)), FACTOR_KEYS, rotation=35, ha='right')
axes[0, 0].set_yticks(range(len(FACTOR_KEYS)), FACTOR_KEYS)
axes[0, 0].set_title(f'{SAMPLE_LABEL}: Mean Daily Spearman Matrix')
for i in range(len(FACTOR_KEYS)):
    for j in range(len(FACTOR_KEYS)):
        axes[0, 0].text(j, i, f'{mean_spearman_matrix.iloc[i, j]:.2f}',
                        ha='center', va='center', fontsize=8)
fig.colorbar(heat, ax=axes[0, 0], fraction=0.046, pad=0.04)

rolling_source = pairwise_daily.pivot(index='date', columns='pair', values='spearman_corr')
rolling_source.index = pd.to_datetime(rolling_source.index)
plot_pairs = pairwise_summary.head(min(10, len(pairwise_summary)))['pair'].tolist()
rolling_source[plot_pairs].rolling(60, min_periods=20).mean().plot(
    ax=axes[0, 1], linewidth=1
)
axes[0, 1].axhline(0, color='black', linewidth=0.8)
axes[0, 1].axhline(MAX_ABS_MEAN_SPEARMAN, color='red', linestyle='--', linewidth=0.8)
axes[0, 1].axhline(-MAX_ABS_MEAN_SPEARMAN, color='red', linestyle='--', linewidth=0.8)
axes[0, 1].set_title('60-day Rolling Spearman (Top |Mean| Pairs)')
axes[0, 1].legend(fontsize=7)

tail_plot = pairwise_summary.set_index('pair')[
    ['mean_long_overlap_min', 'mean_short_overlap_min']
].head(min(15, len(pairwise_summary))).sort_values('mean_long_overlap_min')
tail_plot.plot.barh(ax=axes[1, 0])
axes[1, 0].set_title('Tail Overlap by Factor Pair')

yearly_abs = pairwise_yearly.assign(
    abs_mean_spearman=lambda x: x['mean_spearman'].abs()
).groupby('year')['abs_mean_spearman'].agg(['mean', 'max'])
yearly_abs.plot.bar(ax=axes[1, 1])
axes[1, 1].axhline(MAX_ABS_MEAN_SPEARMAN, color='red', linestyle='--', linewidth=0.8)
axes[1, 1].set_title('Yearly Absolute Pairwise Spearman')
axes[1, 1].tick_params(axis='x', rotation=0)

for ax in axes.flat:
    ax.grid(alpha=0.20)
    ax.set_xlabel('')
plt.tight_layout()
correlation_chart = OUTPUT_DIR / 'multifactor_correlation_dashboard.png'
fig.savefig(correlation_chart, dpi=160, bbox_inches='tight')
plt.show()
print('相关性图表：', correlation_chart)

## 4. 为共同5日回测生成严格入场交易标记

所有单因子和组合使用完全相同的信号日、入场日、退出日与可交易股票交集。

In [ ]:
from backtest import ParquetMarketData, trading_date_offset

market_helper = ParquetMarketData(MARKET_PARQUET, MARKET_CACHE_DIR, ('SH', 'SZ'))
calendar = market_helper.calendar(MIN_STOCKS)
calendar_set = set(calendar)
factor_dates = [date for date in common_factor_dates if date in calendar_set]
candidate_signal_dates = factor_dates[::REBALANCE_STEP]

signal_dates = []
entry_dates = []
for signal_date in candidate_signal_dates:
    entry_date = trading_date_offset(calendar, signal_date, EXECUTION_LAG)
    exit_date = (
        trading_date_offset(calendar, entry_date, HORIZON)
        if entry_date is not None else None
    )
    if entry_date is not None and exit_date is not None and exit_date <= SAMPLE_END_TS:
        signal_dates.append(signal_date)
        entry_dates.append(entry_date)
entry_dates = sorted(set(entry_dates))
entry_date_ints = [int(date.strftime('%Y%m%d')) for date in entry_dates]
assert signal_dates and entry_dates, '样本期内没有可执行的信号/入场日期'

FLAG_COLUMNS = [
    '股票代码', '股票名称', '交易日', '开盘价', '最高价', '最低价', '收盘价',
    '成交量（手）', '成交额（千元）', '复权因子', '当日涨停价', '当日跌停价',
]
manifest_path = FLAGS_DIR / 'manifest.json'

if BUILD_STRICT_FLAGS:
    selected = []
    value_set = pa.array(entry_date_ints)
    parquet_file = pq.ParquetFile(MARKET_PARQUET)
    for batch in parquet_file.iter_batches(
        batch_size=300_000, columns=FLAG_COLUMNS, use_threads=True
    ):
        table = pa.Table.from_batches([batch])
        filtered = table.filter(pc.is_in(table['交易日'], value_set=value_set))
        if filtered.num_rows:
            selected.append(filtered)
    assert selected, '没有读取到入场日行情'
    flags = pa.concat_tables(selected).to_pandas()
    flags = flags.rename(columns={
        '股票代码': 'code', '股票名称': 'name', '交易日': 'date_int',
        '开盘价': 'open', '最高价': 'high', '最低价': 'low', '收盘价': 'close',
        '成交量（手）': 'volume', '成交额（千元）': 'amount',
        '复权因子': 'adj_factor', '当日涨停价': 'limit_up', '当日跌停价': 'limit_down',
    })
    flags = flags[flags['code'].astype(str).str.endswith(('.SH', '.SZ'))].copy()
    flags['date'] = pd.to_datetime(flags.pop('date_int'), format='%Y%m%d')
    numeric_columns = [
        'open', 'high', 'low', 'close', 'volume', 'amount',
        'adj_factor', 'limit_up', 'limit_down',
    ]
    for column in numeric_columns:
        flags[column] = pd.to_numeric(flags[column], errors='coerce')
    valid_market = (
        flags['code'].notna()
        & (flags[['open', 'high', 'low', 'close']] > 0).all(axis=1)
        & (flags['volume'] > 0)
        & (flags['amount'] > 0)
        & (flags['adj_factor'] > 0)
    )
    is_st = flags['name'].fillna('').str.upper().str.contains('ST', regex=False)
    open_at_limit_up = (
        (flags['limit_up'] > 0)
        & (flags['open'] >= flags['limit_up'] - PRICE_TOLERANCE)
    )
    open_at_limit_down = (
        (flags['limit_down'] > 0)
        & (flags['open'] <= flags['limit_down'] + PRICE_TOLERANCE)
    )
    flags['tradable'] = valid_market & (~is_st) & (~open_at_limit_up) & (~open_at_limit_down)
    assert not flags.duplicated(['date', 'code']).any()
    for date, daily in flags.groupby('date', sort=True):
        daily[['code', 'tradable']].to_csv(FLAGS_DIR / f'{date:%Y-%m-%d}.csv', index=False)
    written_dates = sorted(pd.Timestamp(file.stem) for file in FLAGS_DIR.glob('????-??-??.csv'))
    missing_flag_dates = sorted(set(entry_dates) - set(written_dates))
    assert not missing_flag_dates, f'缺少交易标记：{missing_flag_dates[:5]}'
    flag_summary = pd.Series({
        '实际信号日': len(signal_dates),
        '入场标记日': len(entry_dates),
        '标记总行数': len(flags),
        '基础行情不可交易': int((~valid_market).sum()),
        'ST记录': int(is_st.sum()),
        '开盘触及涨停': int(open_at_limit_up.sum()),
        '开盘触及跌停': int(open_at_limit_down.sum()),
        '最终可交易': int(flags['tradable'].sum()),
    })
    display(flag_summary.to_frame('数量'))
    manifest = {
        'factor_set': FACTOR_SET_NAME,
        'sample_stage': SAMPLE_STAGE,
        'sample_start': SAMPLE_START_DATE,
        'sample_end': SAMPLE_END_DATE,
        'horizon': HORIZON,
        'execution_lag': EXECUTION_LAG,
        'rebalance_step': REBALANCE_STEP,
        'price_tolerance': PRICE_TOLERANCE,
        'entry_dates': [f'{date:%Y-%m-%d}' for date in entry_dates],
        'rule': 'valid OHLC/volume/amount/adj_factor; non-ST; open not at price limits',
    }
    manifest_path.write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8'
    )
    del flags, selected
    gc.collect()
else:
    assert manifest_path.exists(), f'找不到交易标记清单：{manifest_path}'
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    expected = {
        'factor_set': FACTOR_SET_NAME,
        'sample_stage': SAMPLE_STAGE,
        'sample_start': SAMPLE_START_DATE,
        'sample_end': SAMPLE_END_DATE,
        'horizon': HORIZON,
        'execution_lag': EXECUTION_LAG,
        'rebalance_step': REBALANCE_STEP,
        'price_tolerance': PRICE_TOLERANCE,
    }
    for key, value in expected.items():
        assert manifest.get(key) == value, f'交易标记缓存参数不一致：{key}'
    for date in entry_dates:
        assert (FLAGS_DIR / f'{date:%Y-%m-%d}.csv').exists(), f'缺少交易标记：{date:%Y-%m-%d}'
    print('使用已有严格交易标记：', FLAGS_DIR)

## 5. 所有单因子、全组合及逐因子剔除组合的共同口径回测

共同回测使用 T 日收盘信号、T+1 VWAP 入场、持有5个交易日、5日非重叠调仓，并计入冻结交易成本。

In [ ]:
from backtest import BacktestConfig, DataPaths, FactorSpec, StrictFactorBacktester

BACKTEST_SIGNAL_COLUMNS = {
    **{cfg['alpha_column']: cfg['alpha_column'] for cfg in FACTOR_CONFIGS.values()},
    **{column: column for column in COMBINATION_WEIGHTS},
}
results = {}
if RUN_STRICT_BACKTEST:
    for signal_name, factor_column in BACKTEST_SIGNAL_COLUMNS.items():
        run_output = OUTPUT_DIR / 'strict_backtests' / signal_name
        paths = DataPaths(
            factor_dir=MULTIFACTOR_PATH,
            market_parquet=MARKET_PARQUET,
            market_cache_dir=MARKET_CACHE_DIR,
            output_dir=run_output,
            flags_dir=FLAGS_DIR,
        )
        factor = FactorSpec(
            name=signal_name,
            column=factor_column,
            direction='positive',
            preprocessed=True,
        )
        config = BacktestConfig(
            start_date=SAMPLE_START_DATE,
            end_date=SAMPLE_END_DATE,
            horizon=HORIZON,
            execution_lag=EXECUTION_LAG,
            rebalance_step=REBALANCE_STEP,
            groups=GROUPS,
            cost_bps=COST_BPS,
            min_stocks=MIN_STOCKS,
            tradability_mode='basic',
            missing_return_policy='zero',
            max_missing_return_ratio=MAX_MISSING_RETURN_RATIO,
            standardize=False,
            neutralize=False,
            market_exchanges=('SH', 'SZ'),
            save_holdings=SAVE_HOLDINGS,
            allow_exit_after_end=False,
        )
        print('\n开始回测：', signal_name)
        results[signal_name] = StrictFactorBacktester(paths, factor, config).run(save=True)
else:
    print('RUN_STRICT_BACKTEST=False：已跳过共同口径回测。')

## 6. IC相关性、策略收益相关性与多因子边际贡献

全组合首先与最佳单因子比较；三因子以上还会自动比较“全组合”与“剔除某因子的组合”，衡量每个因子的边际夏普贡献。

In [ ]:
if RUN_STRICT_BACKTEST:
    performance_rows = []
    ic_frames = []
    return_frames = []
    integrity_rows = []
    for name, result in results.items():
        perf = result.performance.loc['long_short_net']
        performance_rows.append({'signal': name, **perf.to_dict()})
        ic_frames.append(result.ic_series[['rank_ic']].rename(columns={'rank_ic': name}))
        return_frames.append(
            result.group_returns[['long_short_net']].rename(columns={'long_short_net': name})
        )
        integrity_rows.append({
            'signal': name,
            'mean_fill_ratio': result.audit['fill_ratio'].mean(),
            'min_fill_ratio': result.audit['fill_ratio'].min(),
            'mean_missing_return_ratio': result.audit['missing_return_ratio'].mean(),
            'max_missing_return_ratio': result.audit['missing_return_ratio'].max(),
        })

    performance_comparison = pd.DataFrame(performance_rows).set_index('signal')
    common_ic = pd.concat(ic_frames, axis=1, join='inner').dropna()
    common_returns = pd.concat(return_frames, axis=1, join='inner').dropna()
    ic_correlation = common_ic.corr(method='pearson')
    return_correlation = common_returns.corr(method='pearson')
    integrity_comparison = pd.DataFrame(integrity_rows).set_index('signal')

    component_names = list(FACTOR_ALPHA_COLUMNS.values())
    component_best_sharpe = performance_comparison.loc[component_names, 'sharpe'].max()
    combined_sharpe = performance_comparison.loc[COMBINED_COLUMN, 'sharpe']
    sharpe_gain = combined_sharpe - component_best_sharpe
    if sharpe_gain >= MIN_SHARPE_GAIN_FOR_STRONG:
        incremental_effect = '显著'
    elif sharpe_gain >= -0.05:
        incremental_effect = '中等/基本持平'
    else:
        incremental_effect = '微弱/负增益'

    marginal_rows = []
    for factor_key, without_column in LEAVE_ONE_OUT_COLUMNS.items():
        marginal_rows.append({
            'factor': factor_key,
            'full_combination_sharpe': combined_sharpe,
            'without_factor_sharpe': performance_comparison.loc[without_column, 'sharpe'],
            'marginal_sharpe_gain': (
                combined_sharpe - performance_comparison.loc[without_column, 'sharpe']
            ),
            'full_combination_annual_return': performance_comparison.loc[
                COMBINED_COLUMN, 'annual_return'
            ],
            'without_factor_annual_return': performance_comparison.loc[
                without_column, 'annual_return'
            ],
        })
    marginal_contribution = (
        pd.DataFrame(marginal_rows).set_index('factor') if marginal_rows
        else pd.DataFrame(columns=[
            'full_combination_sharpe', 'without_factor_sharpe',
            'marginal_sharpe_gain', 'full_combination_annual_return',
            'without_factor_annual_return',
        ])
    )

    def max_abs_off_diagonal(matrix, names):
        sub = matrix.loc[names, names]
        mask = np.triu(np.ones(sub.shape, dtype=bool), k=1)
        values = sub.where(mask).stack()
        return float(values.abs().max()) if len(values) else np.nan

    max_abs_ic_corr = max_abs_off_diagonal(ic_correlation, component_names)
    max_abs_return_corr = max_abs_off_diagonal(return_correlation, component_names)
    min_marginal_gain = (
        marginal_contribution['marginal_sharpe_gain'].min()
        if not marginal_contribution.empty else np.nan
    )
    positive_marginal_count = (
        int((marginal_contribution['marginal_sharpe_gain'] > 0).sum())
        if not marginal_contribution.empty else 0
    )

    final_summary = pd.Series({
        'sample_stage': SAMPLE_STAGE,
        'sample_start': SAMPLE_START_DATE,
        'sample_end': SAMPLE_END_DATE,
        'factor_count': len(FACTOR_KEYS),
        'worst_cross_sectional_pair': factor_corr_summary['worst_pair'],
        'max_abs_mean_daily_spearman': factor_corr_summary['max_abs_mean_spearman'],
        'factor_correlation_pass': correlation_pass,
        'max_abs_component_ic_correlation': max_abs_ic_corr,
        'max_abs_component_return_correlation': max_abs_return_corr,
        'best_component_net_sharpe': component_best_sharpe,
        'combined_net_sharpe': combined_sharpe,
        'combined_sharpe_gain_vs_best_single': sharpe_gain,
        'incremental_effect_vs_best_single': incremental_effect,
        'minimum_leave_one_out_sharpe_gain': min_marginal_gain,
        'positive_marginal_factor_count': positive_marginal_count,
        'combined_annual_return': performance_comparison.loc[COMBINED_COLUMN, 'annual_return'],
        'combined_max_drawdown': performance_comparison.loc[COMBINED_COLUMN, 'max_drawdown'],
        'combined_min_fill_ratio': integrity_comparison.loc[COMBINED_COLUMN, 'min_fill_ratio'],
        'combined_max_missing_return_ratio': integrity_comparison.loc[
            COMBINED_COLUMN, 'max_missing_return_ratio'
        ],
    })

    performance_comparison.to_csv(OUTPUT_DIR / 'performance_comparison.csv')
    ic_correlation.to_csv(OUTPUT_DIR / 'ic_correlation_matrix.csv')
    return_correlation.to_csv(OUTPUT_DIR / 'strategy_return_correlation_matrix.csv')
    integrity_comparison.to_csv(OUTPUT_DIR / 'integrity_comparison.csv')
    marginal_contribution.to_csv(OUTPUT_DIR / 'marginal_contribution.csv')
    final_summary.to_csv(
        OUTPUT_DIR / 'final_correlation_and_incremental_summary.csv', header=['value']
    )
    display(performance_comparison)
    display(ic_correlation)
    display(return_correlation)
    display(integrity_comparison)
    if not marginal_contribution.empty:
        display(marginal_contribution)
    display(final_summary.to_frame('value'))

    if SAMPLE_STAGE == 'validation':
        train_profile = SAMPLE_PROFILES['train']
        train_tag = (
            f"{pd.Timestamp(train_profile['start_date']):%Y%m%d}_"
            f"{pd.Timestamp(train_profile['end_date']):%Y%m%d}"
        )
        train_summary_path = (
            OUTPUT_ROOT / f'{HORIZON}d_train_{train_tag}'
            / 'final_correlation_and_incremental_summary.csv'
        )
        if train_summary_path.exists():
            train_summary = pd.read_csv(train_summary_path, index_col=0)['value']
            comparison_keys = [
                'max_abs_mean_daily_spearman',
                'max_abs_component_ic_correlation',
                'max_abs_component_return_correlation',
                'best_component_net_sharpe',
                'combined_net_sharpe',
                'combined_sharpe_gain_vs_best_single',
                'minimum_leave_one_out_sharpe_gain',
                'combined_annual_return',
                'combined_max_drawdown',
            ]
            train_validation_comparison = pd.DataFrame({
                'train': train_summary.reindex(comparison_keys),
                'validation': final_summary.reindex(comparison_keys),
            })
            train_validation_comparison.to_csv(
                OUTPUT_DIR / 'train_validation_comparison.csv'
            )
            display(train_validation_comparison)
        else:
            print('尚未找到同一因子集合的训练集汇总，请先完整运行 train 阶段。')

In [ ]:
if RUN_STRICT_BACKTEST:
    nav = (1 + common_returns).cumprod()
    if len(nav.columns) <= 12:
        nav_plot_columns = list(nav.columns)
    else:
        top_signals = performance_comparison['sharpe'].nlargest(10).index.tolist()
        nav_plot_columns = list(dict.fromkeys([COMBINED_COLUMN] + top_signals))

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    nav[nav_plot_columns].plot(ax=axes[0, 0], linewidth=1.1)
    axes[0, 0].axhline(1, color='black', linewidth=0.8)
    axes[0, 0].set_title(f'{SAMPLE_LABEL}: Strict {HORIZON}-day Net NAV')
    axes[0, 0].legend(fontsize=7)

    performance_comparison['sharpe'].sort_values().plot.barh(ax=axes[0, 1])
    axes[0, 1].set_title('Net Sharpe Comparison')

    ic_plot_columns = component_names + [COMBINED_COLUMN]
    common_ic[ic_plot_columns].rolling(20, min_periods=10).mean().plot(
        ax=axes[1, 0], linewidth=1
    )
    axes[1, 0].axhline(0, color='black', linewidth=0.8)
    axes[1, 0].set_title('20-signal Rolling RankIC')
    axes[1, 0].legend(fontsize=7)

    if not marginal_contribution.empty:
        marginal_contribution['marginal_sharpe_gain'].sort_values().plot.barh(
            ax=axes[1, 1]
        )
        axes[1, 1].axvline(0, color='black', linewidth=0.8)
        axes[1, 1].set_title('Leave-one-out Marginal Sharpe Gain')
    else:
        integrity_comparison[['mean_fill_ratio', 'max_missing_return_ratio']].plot.bar(
            ax=axes[1, 1]
        )
        axes[1, 1].set_title('Execution Integrity')
        axes[1, 1].tick_params(axis='x', rotation=15)

    for ax in axes.flat:
        ax.grid(alpha=0.25)
        ax.set_xlabel('')
    plt.tight_layout()
    incremental_chart = OUTPUT_DIR / 'strict_multifactor_dashboard.png'
    fig.savefig(incremental_chart, dpi=160, bbox_inches='tight')
    plt.show()

    protocol = {
        'sample_stage': SAMPLE_STAGE,
        'sample_label': SAMPLE_LABEL,
        'sample_start': SAMPLE_START_DATE,
        'sample_end': SAMPLE_END_DATE,
        'next_start_locked': NEXT_START_LOCKED,
        'factor_set': FACTOR_SET_NAME,
        'factor_sources': {
            key: str(cfg['active_path']) for key, cfg in FACTOR_CONFIGS.items()
        },
        'factor_columns': {
            key: cfg['column'] for key, cfg in FACTOR_CONFIGS.items()
        },
        'factor_directions': {
            key: cfg['direction'] for key, cfg in FACTOR_CONFIGS.items()
        },
        'direction_alignment_signs': {
            key: cfg['sign'] for key, cfg in FACTOR_CONFIGS.items()
        },
        'full_combination_weights': NORMALIZED_WEIGHTS.to_dict(),
        'leave_one_out_columns': LEAVE_ONE_OUT_COLUMNS,
        'combination': 'frozen non-negative weights followed by daily z-score',
        'horizon': HORIZON,
        'execution_lag': EXECUTION_LAG,
        'rebalance_step': REBALANCE_STEP,
        'cost_bps': COST_BPS,
        'strict_entry_flags': True,
        'factor_correlation_pass': bool(correlation_pass),
        'incremental_effect_vs_best_single': incremental_effect,
    }
    (OUTPUT_DIR / 'correlation_protocol.json').write_text(
        json.dumps(protocol, ensure_ascii=False, indent=2), encoding='utf-8'
    )
    print('输出目录：', OUTPUT_DIR)

## 7. 解释规则与以后新增因子的操作顺序

1. 新增因子时，只在 `FACTOR_CONFIGS` 中复制一个配置块，填写唯一英文键、因子列名、训练阶段确定的方向、冻结权重、训练 Parquet 和验证 Parquet。
2. 保持 `SAMPLE_STAGE = 'train'`，重启内核并从头运行。只在训练集决定新因子是否保留以及权重；不要因为训练结果临时改变相关性门槛。
3. 冻结因子集合、方向、权重和共同5日口径后，仅把 `SAMPLE_STAGE` 改为 `'validation'`，重启内核并从头运行。
4. 验证阶段只确认训练结论是否复现，不得根据验证结果调整权重、方向、持有期、分组或因子定义。

重点输出：

- `pairwise_correlation_summary.csv`：每一对因子的横截面相关性和尾部重合；
- `mean_daily_spearman_matrix.csv`：多因子相关矩阵；
- `performance_comparison.csv`：所有单因子和组合的共同口径表现；
- `marginal_contribution.csv`：加入每个因子对全组合夏普的边际贡献；
- `final_correlation_and_incremental_summary.csv`：本阶段总判断；
- `train_validation_comparison.csv`：验证阶段自动读取同一因子集合的训练结果。

边界说明：

- 共同回测只使用全部因子的股票交集，所以单因子结果可能与其独立回测略有不同；
- 因子越多，共同股票池可能越小，必须同时检查 `minimum_common_stocks` 和成交覆盖率；
- 逐因子剔除比较是诊断，不授权在验证集上重新选择权重；
- `flags_dir` 会实际参与 T+1 入场可交易判断；退出日缺失 VWAP 的记录按冻结协议记0并进入审计；
- 当前没有逐笔委托、盘口排队、冲击成本和可借券数据，因此“严格”仍是日频数据可支持的严格程度；
- 训练阶段不会读取验证因子文件，验证阶段不会读取测试期数据，且退出日不得越过本阶段截止日。